# Notebook 12: Verbaliser Diagnostic — Token Anchoring vs Concept Tracking

**Purpose**: Resolve Notebook 11's saturation confound by trying to de-saturate
the model's output, and in doing so identify *why* the label channel looks inert.

## The design

Four verbalisers, everything else held fixed, corruption fixed at 0%:

| arm | tokens | label gloss in system prompt | tests |
|---|---|---|---|
| baseline | `No` / `Yes` | kept | reference |
| **swapped** | `Yes` / `No` | kept, still attached to the same *class indices* | **token- vs concept-anchoring** |
| neutral | `A` / `B` | stripped | Wei et al. 2023 symbol tuning |
| numeric | `0` / `1` | stripped | as above, different token family |

**The swap is the decisive arm.** Only which literal string denotes which class
changes; the semantic meanings stay attached to the same class indices via
`TASK_DESCRIPTIONS`. If the model is reading the demonstrations' semantics, its
*class* predictions should be unchanged — nothing about the task moved. If it is
anchored to the literal token, its predictions invert.

Pre-registered success criterion for de-saturation: some verbaliser gives a
positive rate in [0.35, 0.65] **with IQR >= 0.15**.

```bash
cd sata-project
# one job per config; --label-tokens takes (token_for_class0, token_for_class1)
PYTHONPATH=. python scripts/run_real_arm_grid.py --mode corruption-gate \
    --cache-dir _screen_cache --model Llama-3.1-8B-Instruct \
    --out results/v2/verbaliser_swapped --corruptions 0.0 \
    --label-tokens Yes No --tensor-parallel 1 --max-model-len 4096
# neutral/numeric arms add --strip-label-meanings
```

4 configs x 4 datasets x 5 seeds x 250 queries = 24,000 rows.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

pd.set_option("display.width", 220)
RESULTS = PROJECT_ROOT / "results" / "v2"


def margin(df):
    """Label-logprob margin: the model's decision variable before thresholding."""
    return df.logprob_1 - df.logprob_0


def p_positive(df):
    """Implied P(positive) = sigmoid(margin)."""
    return 1.0 / (1.0 + np.exp(df.logprob_0 - df.logprob_1))


def integrity(df, name=""):
    """A failed label-token lookup defaults to -100 and a failed decode to -1.
    Both would masquerade as findings, so check before interpreting anything."""
    bad_lp = ((df.logprob_0 == -100) | (df.logprob_1 == -100)).mean()
    bad_pred = (df.prediction_raw == -1).mean()
    print(f"{name}invalid predictions: {bad_pred:.5f} | logprob sentinels: {bad_lp:.5f}")
    return bad_lp == 0 and bad_pred == 0

In [3]:
verb = pd.read_parquet(RESULTS / "verbaliser_merged.parquet")
assert integrity(verb)
fs = verb[verb.mechanism == "random"]
print(f"{len(verb)} rows | configs={sorted(verb.verbaliser.unique())}")

invalid predictions: 0.00000 | logprob sentinels: 0.00000
24000 rows | configs=['baseline', 'neutral_ab', 'numeric_01', 'swapped']


## Step 1: The swap inverts the predicted class

`margin` here is **class-oriented**: `logprob_1 - logprob_0` where index 1 is
always the positive *class*, whatever token denotes it. So a model tracking
concepts should give a similar class-oriented margin under the swap.

In [4]:
tab = fs.groupby(["dataset", "verbaliser"]).apply(lambda s: pd.Series({
    "margin_class_oriented": float(margin(s).mean()),
    "posrate_raw": (s.prediction_raw == 1).mean(),
    "bacc_raw": balanced_accuracy_score(s.label, s.prediction_raw),
    "auroc": roc_auc_score(s.label, margin(s)),
}))
tab.unstack().round(3)

margin_class_oriented                               posrate_raw                               bacc_raw                                  auroc                              
verbaliser                  baseline neutral_ab numeric_01 swapped    baseline neutral_ab numeric_01 swapped baseline neutral_ab numeric_01 swapped baseline neutral_ab numeric_01 swapped
dataset                                                                                                                                                                                   
acsincome                      1.891     -0.475      1.078   0.098       0.995      0.289      0.893   0.545    0.502      0.562      0.535   0.518    0.678      0.562      0.632   0.514
acspubcov                      1.859     -0.259      1.248  -0.252       1.000      0.298      0.998   0.127    0.500      0.512      0.502   0.517    0.519      0.511      0.501   0.517
anes                           1.263      0.364      0.613  -0.408       0.927      0.650      0.786   0.153    0.519      0.599      0.582   0.520    0.678      0.659      0.633   0.554
brfss_diabetes                 2.479     -0.735      0.997  -0.541       1.000      0.154      0.995   0.001    0.500      0.537      0.503   0.500    0.548      0.563      0.565   0.470

**The model has a raw lexical preference for the literal string `Yes`** (and,
more weakly, `1`), independent of which class that string is assigned to.
Swapping `No`/`Yes` to `Yes`/`No` collapses the predicted-positive rate from
0.93-1.00 down to 0.00-0.55 on every dataset. The model keeps preferring to emit
"Yes", and because "Yes" now denotes the *negative* class, its class predictions
invert.

Confirmed a second way: `0`/`1` reproduces almost the same failure mode as
`No`/`Yes` — "1" attracts the same affirmative-token preference even though
nothing in the prompt calls it an affirmative answer. `A`/`B` is the only pair
with a mild *opposite* bias.

This is **token anchoring, not concept tracking**, and it is a direct empirical
instance of the mechanism *Semantic Anchors in In-Context Learning: Why Small
LLMs Cannot Flip Their Labels* (arXiv 2511.21038) reports across 1-12B models.

## Step 2: The pre-registered de-saturation criterion fails for all four arms

In [5]:
pyes = fs.assign(pyes=p_positive)
posrate = fs.groupby(["dataset", "verbaliser"]).apply(lambda s: (s.prediction_raw == 1).mean()).unstack()
iqr = pyes.groupby(["dataset", "verbaliser"]).pyes.apply(
    lambda v: np.diff(np.percentile(v, [25, 75]))[0]).unstack()

passes = posrate.apply(lambda col: col.between(0.35, 0.65)) & (iqr >= 0.15)
print("posrate (IQR in brackets):")
print((posrate.round(3).astype(str) + " (" + iqr.round(3).astype(str) + ")").to_string())
print("\npasses [0.35,0.65] AND IQR>=0.15:")
print(passes.to_string())
print("\nany config passing on every dataset:", passes.all(axis=0).any())

posrate (IQR in brackets):
verbaliser           baseline     neutral_ab     numeric_01        swapped
dataset                                                                   
acsincome       0.995 (0.087)  0.289 (0.286)  0.893 (0.184)  0.545 (0.124)
acspubcov         1.0 (0.058)  0.298 (0.183)  0.998 (0.087)  0.127 (0.061)
anes            0.927 (0.174)   0.65 (0.237)  0.786 (0.224)  0.153 (0.169)
brfss_diabetes    1.0 (0.035)  0.154 (0.246)  0.995 (0.098)  0.001 (0.059)

passes [0.35,0.65] AND IQR>=0.15:
verbaliser      baseline  neutral_ab  numeric_01  swapped
dataset                                                  
acsincome          False       False       False    False
acspubcov          False       False       False    False
anes               False       False       False    False
brfss_diabetes     False       False       False    False

any config passing on every dataset: False


No verbaliser lands in the target zone on all four datasets. `A`/`B` gets
closest — and is also the only arm with meaningfully wide dynamic range — but it
is still a failure of the stated criterion.

**AUROC also moves when the semantic gloss is removed**, and not uniformly: it is
*higher* than baseline only on BRFSS, and *lower* on the other three. So the gloss
is not a pure additive bias that calibration would absorb — some of what
natural-language labels contribute is genuine task signal.

## Step 3: The decisive check — does calibration change the gate verdict?

Contextual calibration removes the *static* token bias (Notebook 11 §3). If
Notebook 11's corruption result were an artefact of the saturated threshold, it
should disappear once calibration fixes the threshold. Re-running the original
gate's corruption sweep on **calibrated balanced accuracy**:

In [6]:
gate = pd.read_parquet(RESULTS / "gate_corruption_merged.parquet")
rows = []
for ds, g in gate[gate.mechanism == "random"].groupby("dataset"):
    per = {c: gc.groupby("seed").apply(lambda x: balanced_accuracy_score(x.label, x.prediction))
           for c, gc in g.groupby("corruption")}
    b0, b1 = per[0.0].to_numpy(), per[1.0].to_numpy()
    rows.append(dict(dataset=ds, bacc_c0=b0.mean(), bacc_c50=per[0.5].mean(), bacc_c100=b1.mean(),
                     delta=b0.mean() - b1.mean(), seeds_same_dir=int((b0 > b1).sum()),
                     wilcoxon_p=stats.wilcoxon(b0, b1).pvalue))
pd.DataFrame(rows).sort_values("delta", ascending=False).round(3)

,dataset,bacc_c0,bacc_c50,bacc_c100,delta,seeds_same_dir,wilcoxon_p
2,anes,0.638,0.550,0.516,0.122,5,0.062
1,acspubcov,0.505,0.538,0.523,-0.019,2,0.438
0,acsincome,0.665,0.694,0.685,-0.020,1,0.125
3,brfss_diabetes,0.559,0.575,0.586,-0.027,0,0.062


**Unchanged.** ANES is still the only dataset whose calibrated accuracy responds
to label corruption, with the same 5/5 seeds and the same `p` at the 5-seed floor.

That is three independent metrics agreeing — raw-margin AUROC (Notebook 11 §1),
raw accuracy, and now calibrated balanced accuracy. The label-channel inertness
on three of four datasets is a **real property of this model at this scale**, not
a measurement artefact.

## Revised diagnosis

Neither of Notebook 11's two hypotheses was right on its own. The label channel is
not "inert" in a simple sense — it is *masked* by a token-level lexical prior
strong enough that no natural single-token verbaliser fully de-saturates it. But
calibration corrects the aggregate bias and the corruption verdict survives
regardless, so the inertness conclusion stands, with token anchoring as the
mechanism.

Notebook 13 settles it properly, with a model whose output is 6.5x less
saturated.